# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template and practical guide for loading and exploring the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
Data is defined by a Croissant schema and can be accessed via:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Record Sets in this dataset:", [rs['@id'] for rs in metadata.record_sets])

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their fields and columns
record_sets = metadata.record_sets
for rs in record_sets:
    print("Record Set @id:", rs['@id'])
    print("  Name:", rs['name'] if 'name' in rs else rs['@id'])
    if 'fields' in rs:
        print("  Fields:")
        for field in rs['fields']:
            print("    Field @id:", field['@id'], "  | Name:", field.get('name', field['@id']), "| DataType:", field.get('dataType'))
    if 'columns' in rs:
        print("  Columns:")
        for column in rs['columns']:
            print("    Column @id:", column['@id'], "  | Name:", column.get('name', column['@id']), "| DataType:", column.get('dataType'))
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
All entities are referenced by their `@id` fields to ensure consistency.

In [ ]:
# Compile all record set @ids
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Load all records for this record set
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"\nRecord set '{rs_id}' columns:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Explore, filter, and transform the dataset. Use fields referenced by their `@id`.

Here, we'll select a numeric field from Table 2 (e.g., hormone levels), filter patients with a hormone measurement above a threshold, normalize the values, and optionally group by a clinical attribute from Table 1.

In [ ]:
# Find an appropriate numeric field @id (for demonstration, replace below with actual @id)
# Let's assume record set Table 2 has hormone field called 'cr:field_Hormone_Level' and patient id from Table 1

table2_rs_id = None
table2_field_id = None

# Search record set and field for hormone measurement in Table 2
for rs in metadata.record_sets:
    if 'hormone' in (rs.get('name', '')).lower() or 'Table 2' in rs.get('name', ''):
        table2_rs_id = rs['@id']
        for field in rs.get('fields', []):
            if 'hormone' in field.get('name', '').lower() or 'level' in field.get('name', '').lower():
                table2_field_id = field['@id']
                break
        if table2_field_id: break

# If not found, fallback to first numeric field in the first record set
if not table2_rs_id or not table2_field_id:
    for rs in metadata.record_sets:
        for field in rs.get('fields', []):
            if field.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']:
                table2_rs_id = rs['@id']
                table2_field_id = field['@id']
                break
        if table2_field_id: break

print(f"Using record set @id: {table2_rs_id}, field @id: {table2_field_id} for EDA.")

# Define threshold for hormone/measurement
threshold = 10
df2 = dataframes[table2_rs_id]

# Filtering rows with value > threshold
if table2_field_id in df2.columns:
    filtered_df = df2[df2[table2_field_id] > threshold]
    print(f"Filtered records with {table2_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the field
    norm_col = f"{table2_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[table2_field_id] - filtered_df[table2_field_id].mean()) / filtered_df[table2_field_id].std()
    print(f"Normalized {table2_field_id} for filtered records:")
    print(filtered_df[[table2_field_id, norm_col]].head())

    # Optionally group by a clinical attribute from Table 1
    # Find a categorical/group field in Table 1
    group_field = None
    table1_rs_id = None
    for rs in metadata.record_sets:
        if 'clinical' in (rs.get('name', '').lower()) or 'Table 1' in rs.get('name', ''):
            table1_rs_id = rs['@id']
            for field in rs.get('fields', []) + rs.get('columns', []):
                if field.get('dataType') == 'schema:Text' and field.get('name') not in ['Patient ID','id','record_id']:
                    group_field = field['@id']
                    break
            if group_field: break

    if group_field and group_field in df2.columns:
        grouped_df = filtered_df.groupby(group_field)[table2_field_id].mean()
        print(f"Grouped data by {group_field} (mean {table2_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found in columns for EDA.")

## 5. Visualization
Visualize the numeric field distribution for the hormone levels in Table 2 using matplotlib or seaborn. All plotting refers to field and record set by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df2 = dataframes[table2_rs_id]

# Simple histogram of hormone levels
if table2_field_id in df2.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df2[table2_field_id].dropna(), bins=8, kde=True)
    plt.title(f"Distribution of '{table2_field_id}' in record set '{table2_rs_id}'")
    plt.xlabel(table2_field_id)
    plt.ylabel('Count')
    plt.show()

    # Scatter plot if another numeric field available
    other_numeric = [col for col in df2.columns if col != table2_field_id and str(df2[col].dtype)[:3] in ['int','flo']]
    if other_numeric:
        plt.figure(figsize=(7,4))
        sns.scatterplot(x=df2[other_numeric[0]], y=df2[table2_field_id])
        plt.title(f"{table2_field_id} vs {other_numeric[0]} in record set {table2_rs_id}")
        plt.xlabel(other_numeric[0])
        plt.ylabel(table2_field_id)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated FAIR data exploration using `mlcroissant`, referencing all entities by their `@id` fields for clarity and reproducibility.

- Loaded metadata and overviewed record sets, fields, and columns
- Extracted tables using `mlcroissant` and displayed field contents
- Performed EDA on hormone measurements, including filtering and normalization
- Visualized distributions using code referencing dataset entity `@id`s
- You may extend this notebook further for custom clinical or research analyses.

Always cite according to FAIR and the dataset's specified `citeAs`.